# Brief 01 - "remaniement" complet

In [2]:
import os
from pathlib import Path
import subprocess
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values 

> ⚠️ Il faut faire attention à bien modifier la variable `RELEVES_ROOT` en fonction de l'architecture de votre dossier / repository.

In [3]:
RELEVES_ROOT = "releves"
# On ne sélectionne que les dossier grâce à l'usage de `.is_dir()` 
SITES = sorted(d.name for d in Path(RELEVES_ROOT).iterdir() if d.is_dir())
print(SITES)

['animalis', 'bitiba_fr', 'chronovet', 'clubvetshop', 'maxizoo', 'pharmacy4pets', 'univers_veto', 'vetoplus', 'vetostore', 'zooplus_fr']


In [4]:
# ⚠️ Il faut faire attention à bien modifier la variable DATABASE_URL, avec vos valeurs
conn = psycopg2.connect(
    dbname="vetprice",
    user="postgres",
    password="Mkilo1990",
    host="localhost",
    port=5432
)
conn.autocommit = False
cur = conn.cursor()

## 1. Profilage et modélisation

### Organisation des données

Chaque site a été scrapé plusieurs fois : 
- une fois en avril
- deux en juillet

Un "run" (i.e. une exécution d'un scraper pour un site) produit un dossier horodaté `AAAA-MM-JJ_HHMM`, qui contient les produits collectés ce jour-là. Rien n'est écrasé d'un run à l'autre : 🔥🔥🔥 **c'est ce qui rend l'historisation possible**.

```
releves/
├── chronovet/
│   ├── 2026-04-01_1919/          run du 1er avril à 19h19
│   ├── 2026-07-12_1150/          run du 12 juillet à 11h50  (partiel car interrompu)
│   ├── 2026-07-12_1242/          run du 12 juillet à 12h42  (complet)
│   └── 2026-07-18_1334/          run du 18 juillet à 13h34
├── univers_veto/
│   ├── 2026-04-01_1847/
│   ├── 2026-04-01_1918/
│   ├── 2026-07-12_1148/
│   ├── 2026-07-12_1150/
│   ├── 2026-07-12_1242/
│   ├── 2026-07-18_1334/
│   └── 2026-07-18_1355/
├── animalis/    …
├── bitiba_fr/   …
├── clubvetshop/ …
├── maxizoo/     …
├── pharmacy4pets/ …
├── vetoplus/    …
├── vetostore/   …
└── zooplus_fr/  …
```

Un run contient : 

```
2026-07-18_1334/
├── products.jsonl    un produit par ligne, au format JSON (le champ scraped_at porte l'instant de collecte)
└── meta.json         pas toujours présent, métadonnées du run (products_scraped, http_requests, http_errors, completed, resumed)
```

**Pourquoi y a-t-il plusieurs dossiers le même jour ?**

- Un run peut être interrompu (site lent, coupure, mise en veille de l'ordinateur portable) puis relancé.
- Chaque relance (i.e. nouveau run) crée un nouveau dossier horodaté (i.e. le dossier du run).
- Par exemple, sur chronovet le 12 juillet, `_1150` est un run partiel et `_1242` un run complet du même jour.
- Le champ `HHMM` du nom permet de savoir lequel est le dernier


### Nombre de fichiers de relevés (i.e. un fichier par run)

In [5]:
RELEVES = [
    (site, run_dir / "products.jsonl")
    for site in SITES
    for run_dir in sorted((Path(RELEVES_ROOT) / site).iterdir())
    if (run_dir / "products.jsonl").exists()
]
print(len(RELEVES), "relevés (fichiers) au total")

41 relevés (fichiers) au total


### Profiler tous les fichiers

In [6]:
# Profilage sur TOUS les fichiers : chaque run de chaque site
def profiler_tous_les_fichiers():
    rows = []
    for site in SITES:
        for run_dir in (Path(RELEVES_ROOT) / site).iterdir():
            f = run_dir / "products.jsonl"
            # S'il n'y a pas de fichier pour ce dossier de run
            # on passe au `run_dir_ suivant dans la boucle (avec `continue`)
            if not f.exists():
                continue
            df = pd.read_json(f, lines=True, dtype=False)
            n = len(df)
            # # ean absent : soit colonne manquante, soit valeur vide (ou NaN, donc à bien vérifier)
            if "ean" in df.columns:
                ean = df["ean"]
                # Ci-dessous, ce n'est pas la façon la plus propre, 
                # mais cette écriture permet d'éviter une étape préliminaire de cleaning
                a_ean = ean.notna() & (ean.astype(str).str.strip() != "")
            else:
                ean, a_ean = pd.Series([None] * n, dtype=object), pd.Series([False] * n)
            rows.append({
                "site": site,
                "run": run_dir.name,
                "lignes": n,
                "sans_ean": int((~a_ean).sum()),
                "doublons_ean": int(ean[a_ean].duplicated().sum()),  # doublons internes au run
                "colonnes": df.shape[1],
            })
    return pd.DataFrame(rows)

In [7]:
stats = profiler_tous_les_fichiers()

# 💫 On ajoute le pourcentage de lignes sans ean par run
stats["sans_ean_%"] = (100 * stats["sans_ean"] / stats["lignes"]).round(1)
print(stats.to_string(index=False))

         site             run  lignes  sans_ean  doublons_ean  colonnes  sans_ean_%
     animalis 2026-04-01_1919    2866         4           634        17         0.1
     animalis 2026-07-12_1150     775         0           148        17         0.0
     animalis 2026-07-12_1242   10782         7          3434        17         0.1
     animalis 2026-07-18_1334   53561       820         31114        17         1.5
    bitiba_fr 2026-04-01_1921   21008         0          8062        20         0.0
    bitiba_fr 2026-07-12_1242   12763         0             0        12         0.0
    bitiba_fr 2026-07-18_1334   12659         0             0        12         0.0
    chronovet 2026-04-01_1919    1631         0             0        14         0.0
    chronovet 2026-07-12_1150      90         0             0        19         0.0
    chronovet 2026-07-12_1242    2436         5             0        19         0.2
    chronovet 2026-07-18_1334    2453         5             0        19     

### Agrégat par site (sur tous les runs)

In [8]:
agg = stats.groupby("site")[["lignes", "sans_ean", "doublons_ean"]].sum()
agg

,lignes,sans_ean,doublons_ean
site,,,
animalis,67984,831,35330
bitiba_fr,46430,0,8062
chronovet,6610,10,0
clubvetshop,18524,10581,8
maxizoo,18675,1017,1
pharmacy4pets,4120,4120,0
univers_veto,2146,135,0
vetoplus,7367,643,147
vetostore,10585,10585,0


### On calcule le pourcentage total de lignes sans ean

In [9]:
profils = {
    s: (int(r.lignes), r.sans_ean / r.lignes, int(r.doublons_ean)) 
    for s, r in agg.iterrows()
}


print(f"\nTotal : {int(stats['lignes'].sum())} lignes sur {len(stats)} fichiers, "
      f"{100 * agg['sans_ean'].sum() / agg['lignes'].sum():.1f}% sans ean global")


Total : 388784 lignes sur 41 fichiers, 7.2% sans ean global


### Il y a 7.2 % d'ean manquants -> il faudra une clef de repli
- ✅ on crée donc immédiatement une variable clé que l'on créera avant l'insertion

## 2. Chargement des relevés

## Toutes les données dans un seul dataframe

In [10]:
# On lit TOUTES les données (tous les runs de tous les sites)
frames = []
for site, chemin in RELEVES:
    d = pd.read_json(chemin, lines=True, dtype=False)
    d["date_releve"] = chemin.parent.name[:10]   # on sélectionne : AAAA-MM-JJ, la date du run
    frames.append(d)
df = pd.concat(frames, ignore_index=True)
print(len(df), "lignes chargées, tous sites et toutes dates confondus")

388784 lignes chargées, tous sites et toutes dates confondus


In [11]:
# Clé produit : l'ean s'il est présent et non vide, sinon l'url.
def cle_produit(row):
    ean = row.get("ean")
    if isinstance(ean, str) and ean.strip():
        return ean
    # Valeur par défaut (i.e. la solution de repli qu'on a choisi)
    return row["url"]

df["cle"] = df.apply(cle_produit, axis=1)
df[["site", "date_releve", "ean", "url", "cle"]].head()

,site,date_releve,ean,url,cle
0,animalis,2026-04-01,4015110034865,https://www.animalis.com/moser-tondeuse-1400-p...,4015110034865
1,animalis,2026-04-01,3182550702874,https://www.animalis.com/royal-canin-croquette...,3182550702874
2,animalis,2026-04-01,4014162613790,https://www.animalis.com/jbl-gant-de-nettoyage...,4014162613790
3,animalis,2026-04-01,4004218758865,https://www.animalis.com/tetra-traitement-gold...,4004218758865
4,animalis,2026-04-01,8010690030661,https://www.animalis.com/ferplast-roue-en-plas...,8010690030661


### Pourquoi `scraped_at` plutôt que `date_releve` ?

🔥 **Avant toute chose, ce n'est pas important de choisir l'un ou l'autre, les explications ci-dessous sont juste fournies à des fins de clarté**

- `date_releve` vient du nom du dossier (c'est le "slice" qui contient les 10 premiers caractères), il y en a un par dossier de run. `scraped_at` est difficile plus difficile à définir : pour faire simple : il y a **plus de `scraped_at` distincts que de `date_releve`**, surtout à cause de la reprise sur interruption : un run interrompu puis relancé réécrit dans le **même dossier** (donc même `date_releve`), mais les produits repris portent un **nouveau `scraped_at`**.

```
dossier  animalis/2026-07-18_1334/     ->  date_releve = "2026-07-18"  (10 premiers caractères)

products.jsonl :
   produit      1   scraped_at = 2026-07-18T13:34:00Z   ┐  run lancé à 13h34
   ...                                                   │  (629 lignes)
   produit    629   scraped_at = 2026-07-18T13:34:00Z   ┘
   ─── interruption (mise en veille), puis reprise ───
   produit    630   scraped_at = 2026-07-18T13:55:43Z   ┐  repris à 13h55,
   ...                                                   │  écrit dans le MÊME dossier
   produit  53561   scraped_at = 2026-07-18T13:55:43Z   ┘  (52 932 lignes)

   => 1 dossier, 1 date_releve, mais 2 scraped_at
```

On va donc choisir de dédoubloner sur `scraped_at` : deux observations à deux instants différents comptent comme deux versions, même si elles sont dans le même dossier.

In [12]:
# Table cible (on repart propre à chaque exécution).
conn.rollback()  # dé-avorte la transaction si une cellule précédente a échoué
cur.execute("DROP TABLE IF EXISTS produit_historise")
cur.execute("""
CREATE TABLE produit_historise (
    site text, cle text, ean text, url text, name text, brand text,
    price numeric, in_stock boolean,
    valid_from timestamptz, valid_to timestamptz, is_current boolean,
    PRIMARY KEY (site, cle, valid_from)
)
""")
conn.commit()

# scraped_at partout : clé de dédoublonnage ET borne temporelle.
# Une version par (site, cle, scraped_at). Deux observations à deux instants
# différents sont deux versions, même dans le même dossier de run.
h = (df.sort_values("scraped_at")
       .drop_duplicates(["site", "cle", "scraped_at"], keep="last")
       .sort_values(["site", "cle", "scraped_at"])
       .copy())
h["valid_from"] = h["scraped_at"]
h["valid_to"] = h.groupby(["site", "cle"])["scraped_at"].shift(-1)  # scraped_at du relevé suivant du produit
h["is_current"] = h["valid_to"].isna()                             # dernière version connue

colonnes = ["site", "cle", "ean", "url", "name", "brand",
            "price", "in_stock", "valid_from", "valid_to", "is_current"]
for c in colonnes:
    if c not in h.columns:
        h[c] = None
donnees = h[colonnes].astype(object).where(pd.notnull(h[colonnes]), None)  # NaT/NaN -> None
lignes = list(donnees.itertuples(index=False, name=None))
execute_values(cur, f"INSERT INTO produit_historise ({','.join(colonnes)}) VALUES %s", lignes)
conn.commit()
print(len(lignes), "versions insérées dans produit_historise")

331056 versions insérées dans produit_historise


In [13]:
# Stat : combien de versions perd-on si on dédoublonne sur date_releve au lieu de scraped_at ?
n_scraped = df.drop_duplicates(["site", "cle", "scraped_at"]).shape[0]
n_date = df.drop_duplicates(["site", "cle", "date_releve"]).shape[0]
print(f"dédup scraped_at  : {n_scraped} versions  (choix retenu)")
print(f"dédup date_releve : {n_date} versions")
print(f"-> date_releve en garderait {n_scraped - n_date} de moins, "
      f"soit -{100 * (n_scraped - n_date) / n_scraped:.1f}%")

dédup scraped_at  : 331056 versions  (choix retenu)
dédup date_releve : 325677 versions
-> date_releve en garderait 5379 de moins, soit -1.6%


## Questions d'entraînement 

- Répondre aux questions ci-dessous avec SQL et pandas (i.e. deux implémentations pour chaque)
- Vous n'avez besoin que de la table `produit_historise` et ce notebook.

In [14]:
import warnings
warnings.filterwarnings("ignore", message=".*SQLAlchemy.*")  # silence l'avertissement pd.read_sql
ph = pd.read_sql("SELECT * FROM produit_historise", conn)

ph["price"] = pd.to_numeric(ph["price"])  # conversion numeric SQL vers float pandas
print(ph.shape)

(331056, 11)


### Q1. (exemple) Combien de produits au catalogue aujourd'hui, par site ?

In [15]:
# --- SQL ---
display(pd.read_sql("""
    SELECT site, count(*) AS n
    FROM produit_historise WHERE is_current
    GROUP BY site ORDER BY n DESC
""", conn))

# --- pandas ---
display(ph[ph.is_current].groupby("site").size()
          .sort_values(ascending=False).rename("n").reset_index())

,site,n
0,zooplus_fr,74975
1,animalis,22463
2,bitiba_fr,13634
3,clubvetshop,8564
4,maxizoo,8483
5,vetostore,5798
6,vetoplus,2810
7,chronovet,2488
8,pharmacy4pets,1378
9,univers_veto,354


,site,n
0,zooplus_fr,74975
1,animalis,22463
2,bitiba_fr,13634
3,clubvetshop,8564
4,maxizoo,8483
5,vetostore,5798
6,vetoplus,2810
7,chronovet,2488
8,pharmacy4pets,1378
9,univers_veto,354


### Q2. Prix actuel minimum, maximum et moyen par site.

In [16]:
# --- SQL ---
print("SQL==>")
display(pd.read_sql("""
    SELECT
        site,
        MIN(price) AS price_min,
        MAX(price) AS price_max,
        AVG(price) AS price_avg
    FROM produit_historise
    WHERE is_current
    GROUP BY site
    ORDER BY site
""", conn))


# --- pandas ---
print("Pandas ==>")
q2 = (ph[ph["is_current"]]
      .groupby("site")["price"]
      .agg(price_min="min", price_max="max", price_avg="mean")
      .reset_index())
display(q2)



SQL==>


,site,price_min,price_max,price_avg
0,animalis,0.32,10799.00,78.396280
1,bitiba_fr,0.29,882.99,34.031856
2,chronovet,1.80,703.20,38.442306
3,clubvetshop,0.16,845.52,31.614401
4,maxizoo,NaN,NaN,NaN
5,pharmacy4pets,0.00,371.33,22.052184
6,univers_veto,0.38,329.48,39.272260
7,vetoplus,0.00,2463.60,35.796747
8,vetostore,0.00,717.59,30.919884
9,zooplus_fr,0.00,999.00,48.723369


Pandas ==>


,site,price_min,price_max,price_avg
0,animalis,0.32,10799.00,78.396280
1,bitiba_fr,0.29,882.99,34.031856
2,chronovet,1.80,703.20,38.442306
3,clubvetshop,0.16,845.52,31.614401
4,maxizoo,NaN,NaN,NaN
5,pharmacy4pets,0.00,371.33,22.052184
6,univers_veto,0.38,329.48,39.272260
7,vetoplus,0.00,2463.60,35.796747
8,vetostore,0.00,717.59,30.919884
9,zooplus_fr,0.00,999.00,48.723369


### Q3. Combien de produits en rupture (`in_stock = false`) actuellement, par site ?

In [17]:
# --- SQL ---
print("SQL ==>")
display(pd.read_sql("""
    SELECT site, COUNT(*) AS n_out_of_stock
    FROM produit_historise
    WHERE is_current AND in_stock = FALSE
    GROUP BY site
    ORDER BY n_out_of_stock DESC
""", conn))

# --- pandas ---
print("Pandas ==>")
q3 = (ph[(ph["is_current"]) & (ph["in_stock"] == False)]
      .groupby("site")
      .size()
      .reset_index(name="n_out_of_stock")
      .sort_values("n_out_of_stock", ascending=False))
display(q3)


SQL ==>


,site,n_out_of_stock
0,zooplus_fr,15712
1,animalis,10043
2,vetoplus,2196
3,vetostore,813
4,bitiba_fr,539
5,pharmacy4pets,174
6,clubvetshop,18
7,univers_veto,13
8,chronovet,5


Pandas ==>


,site,n_out_of_stock
8,zooplus_fr,15712
0,animalis,10043
6,vetoplus,2196
7,vetostore,813
1,bitiba_fr,539
4,pharmacy4pets,174
3,clubvetshop,18
5,univers_veto,13
2,chronovet,5


### Q4. L'historique complet d'un produit : toutes ses versions triées.

In [18]:
PRODUIT = ("vetoplus", "3552791071358")  
site_p, cle_p = PRODUIT

# --- SQL ---
print("SQL ==>")

display(pd.read_sql(f"""
    SELECT *
    FROM produit_historise
    WHERE site = '{site_p}'
      AND cle = '{cle_p}'
    ORDER BY valid_from
""", conn))

# --- pandas ---
print("Pandas ==>")
q4 = ph[(ph["site"] == site_p) & (ph["cle"] == cle_p)] \
        .sort_values("valid_from")
display(q4)


SQL ==>


,site,cle,ean,url,name,brand,price,in_stock,valid_from,valid_to,is_current
0,vetoplus,3552791071358,3552791071358,https://www.veto-plus.fr/divers/3000-seringue-...,Seringue à Insuline Standard X100,None,23.99,True,2026-07-12 14:59:19.001665+00:00,2026-07-18 16:16:15.119687+00:00,False
1,vetoplus,3552791071358,3552791071358,https://www.veto-plus.fr/divers/3000-seringue-...,Seringue à Insuline Standard X100,None,19.99,True,2026-07-18 16:16:15.119687+00:00,NaT,True


Pandas ==>


,site,cle,ean,url,name,brand,price,in_stock,valid_from,valid_to,is_current
123249,vetoplus,3552791071358,3552791071358,https://www.veto-plus.fr/divers/3000-seringue-...,Seringue à Insuline Standard X100,NaN,23.99,True,2026-07-12 14:59:19.001665+00:00,2026-07-18 16:16:15.119687+00:00,False
123250,vetoplus,3552791071358,3552791071358,https://www.veto-plus.fr/divers/3000-seringue-...,Seringue à Insuline Standard X100,NaN,19.99,True,2026-07-18 16:16:15.119687+00:00,NaT,True


### Q5. Quels produits ont le plus « bougé » (le plus de versions PAR site) ?

In [19]:
# --- SQL ---
print("SQL ==>")
display(pd.read_sql("""
    SELECT site, cle, COUNT(*) AS n_versions
    FROM produit_historise
    GROUP BY site, cle
    ORDER BY n_versions DESC
    LIMIT 10
""", conn))

# --- pandas ---
print("Pandas ==>")
q5 = (ph.groupby(["site", "cle"])
        .size()
        .reset_index(name="n_versions")
        .sort_values("n_versions", ascending=False)
        .head(10))
display(q5)


SQL ==>


,site,cle,n_versions
0,vetoplus,3760252080157,12
1,vetoplus,3661716109165,9
2,vetoplus,3661716103064,9
3,univers_veto,3662852002167,7
4,univers_veto,3401144404926,7
5,univers_veto,3700525037976,7
6,univers_veto,3605874344075,7
7,univers_veto,3700454500800,7
8,univers_veto,3352712007356,7
9,univers_veto,3661103071235,7


Pandas ==>


,site,cle,n_versions
58816,vetoplus,3760252080157,12
58229,vetoplus,3661716103064,9
58311,vetoplus,3661716109165,9
57227,univers_veto,3662952002204,7
57021,univers_veto,3352712004041,7
57022,univers_veto,3352712006427,7
57037,univers_veto,3352712007356,7
57038,univers_veto,3352712007400,7
57039,univers_veto,3352712007424,7
57040,univers_veto,3352712007561,7


### Q6. Quel était le prix de tel produit à une date donnée `D` ?

In [20]:
D = "2026-07-15"
PRODUIT = ("vetoplus", "3552791071358")
site_p, cle_p = PRODUIT

# --- SQL ---
print("SQL ==>")
display(pd.read_sql(f"""
    SELECT site, cle, price, valid_from, valid_to
    FROM produit_historise
    WHERE site = '{site_p}'
      AND cle = '{cle_p}'
      AND valid_from <= DATE '{D}'
      AND (valid_to IS NULL OR valid_to > DATE '{D}')
""", conn))

# --- pandas ---
print("Pandas ==>")

#  rendre la date timezone-aware
# D_dt = pd.to_datetime(D).tz_localize("UTC")

q6 = ph[(ph["site"] == site_p) &
         (ph["cle"] == cle_p) &
         (ph["valid_from"] <= D) &
        ((ph["valid_to"].isna()) | (ph["valid_to"] > D))]

display(q6[["site", "cle", "price", "valid_from", "valid_to"]])


SQL ==>


,site,cle,price,valid_from,valid_to
0,vetoplus,3552791071358,23.99,2026-07-12 14:59:19.001665+00:00,2026-07-18 16:16:15.119687+00:00


Pandas ==>


,site,cle,price,valid_from,valid_to
123249,vetoplus,3552791071358,23.99,2026-07-12 14:59:19.001665+00:00,2026-07-18 16:16:15.119687+00:00


### Q7. Combien de produits au catalogue (catalogue composé de tous les produits de tous les sites, pas besoin de `groupby`) à une date passée `D` ?

- Utiliser `valid_from` et `valid_to` dans une clause `WHERE`, ainsi que la date `D`.

In [27]:
D = "2026-07-12"

# --- SQL ---
display(pd.read_sql(f"""
    SELECT site, cle, price, valid_from, valid_to
    FROM produit_historise
    WHERE 

       valid_from <= DATE '{D}'
      AND (valid_to IS NULL OR valid_to > DATE '{D}')
""", conn))
# --- pandas ---

display(
    ph[
        (ph["valid_from"] <= D)
        & (
            ph["valid_to"].isna()
            | (ph["valid_to"] > D)
        )
    ][["site", "cle", "price", "valid_from", "valid_to"]]
)

,site,cle,price,valid_from,valid_to
0,animalis,0000003397510,31.95,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
1,animalis,0015561102285,19.21,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
2,animalis,0015561102469,7.40,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
3,animalis,0015561102483,11.60,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
4,animalis,0015561113762,8.99,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
...,...,...,...,...,...
86498,zooplus_fr,9495603317760,132.00,2026-04-01 19:19:58.422887+00:00,2026-07-12 12:42:29.290926+00:00
86499,zooplus_fr,96543214,21.95,2026-04-01 19:19:58.422887+00:00,2026-07-12 12:42:29.290926+00:00
86500,zooplus_fr,9780470059029,49.99,2026-04-01 19:19:58.422887+00:00,2026-07-12 12:42:29.290926+00:00
86501,zooplus_fr,9780470059036,49.99,2026-04-01 19:19:58.422887+00:00,NaT


,site,cle,price,valid_from,valid_to
2,animalis,0000003397510,31.95,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
22,animalis,0015561102285,19.21,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
25,animalis,0015561102469,7.40,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
28,animalis,0015561102483,11.60,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
31,animalis,0015561113762,8.99,2026-04-01 19:19:56.048505+00:00,2026-07-12 12:42:29.292547+00:00
...,...,...,...,...,...
331045,zooplus_fr,9495603317760,132.00,2026-04-01 19:19:58.422887+00:00,2026-07-12 12:42:29.290926+00:00
331048,zooplus_fr,96543214,21.95,2026-04-01 19:19:58.422887+00:00,2026-07-12 12:42:29.290926+00:00
331051,zooplus_fr,9780470059029,49.99,2026-04-01 19:19:58.422887+00:00,2026-07-12 12:42:29.290926+00:00
331054,zooplus_fr,9780470059036,49.99,2026-04-01 19:19:58.422887+00:00,NaT


### Q8. (DIFFICILE) Top 10 des plus fortes variations de prix entre la première et la dernière version d'un produit pour un même site (euros et %).

In [22]:
# --- SQL ---

# --- pandas ---


### Q9. Pour chaque produit, la variation de prix d'une version à la suivante (`LAG`).

Quand le prix ne bouge pas entre deux relevés (i.e. deux observations), la variation vaut 0.

In [23]:
# --- SQL ---

# --- pandas ---


### Q10. (DIFFICILE) Matching inter-sites : pour un même `ean`, quel écart de prix entre sites aujourd'hui ?

- Attention, il sera difficile d'interpéter toutes les valeurs à ce stade, en particulier quand le même `ean` est utilisé pour un produit et des lots de ce produit.

In [24]:
# --- SQL ---

# --- pandas ---
